In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1

/content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1


In [ ]:
!pip install transformers evaluate datasets
!pip install accelerate sentencepiece
!pip install lm-evaluation-harness


In [ ]:
!pip install "lm-evaluation-harness[math,ifeval,sentencepiece]"
!pip install transformers accelerate datasets evaluate sentencepiece sacrebleu


In [ ]:
!mkdir -p /content/custom_tasks/squad_v1
!cp /content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1/custom_tasks/squad_v1/* /content/custom_tasks/squad_v1/

cp: -r not specified; omitting directory '/content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1/custom_tasks/squad_v1/__pycache__'


In [ ]:
from lm_eval.tasks import TaskManager
tm = TaskManager(include_path="/content/custom_tasks",
                 include_defaults = False)
print(list(tm.all_tasks)[:20])  # should list squadv1


['squadv1']


In [ ]:
!pip install -e ".[dev]"

In [ ]:
!rm -r /content/custom_tasks/

In [ ]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [ ]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
import lm_eval
from lm_eval.tasks import TaskManager

# if your squadv1.yaml is inside the installed lm_eval/tasks tree,
# you don't need include_path
tm = TaskManager()

print("squadv1" in tm.all_tasks)      # or whatever name you put in `task:` in the yaml

True


In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager

tm = TaskManager(
    include_path="/content/drive/MyDrive/TUM/Praktikum/lm-evaluation-harnessSquadV1.1/lm_eval/tasks/squad_v1"
)

model = HFLM(pretrained="facebook/opt-125m")

results = lm_eval.simple_evaluate(
    model,
    tasks=["squadv1"],   # this is the `task:` name in your YAML
    num_fewshot=0,
    batch_size=64,
    task_manager=tm,     # optional if your task is inside the installed lm_eval tree
)

print(results["results"])


INFO - NumExpr defaulting to 12 threads.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.


In [ ]:
import squad_v1.task
importlib.reload(squad_v1.task)

In [ ]:
!pip install sqlitedict

  Preparing metadata (setup.py) ... done
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=07ee593796180c66bdd1b8d6a6f60a6e9cb1ffa9fc30e7301a19db58c208816f
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21/fc016aef45ffcabe27129a2252f061387cbf278d2086225a64
Successfully built sqlitedict


In [ ]:
#USE THIS ONE THIS TIME, it should work!!!
import os
import json
import logging
import torch
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

logger.info("Starting SQuADv1 evaluation script")

# 1. Load custom task from LOCAL disk (not Drive)
TASK_PATH = "/content/custom_tasks/squad_v1"
logger.info(f"Initializing TaskManager with include_path={TASK_PATH}")
tm = TaskManager(include_path=TASK_PATH, include_defaults = False)

task_names = ["squadv1"]
#logger.info(f"Registered tasks: {list(tm.all_tasks.keys())}")
logger.info(f"Will evaluate on tasks: {task_names}")

# 2. Load model on GPU
MODEL_NAME = "facebook/opt-125m"
logger.info(f"Loading model: {MODEL_NAME}")

model = HFLM(pretrained=MODEL_NAME, device="cuda")

print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.model.parameters()).device)

logger.info("Model loaded successfully")

# 3. Run evaluation
logger.info("Running lm_eval.simple_evaluate...")

results = lm_eval.simple_evaluate(
    model,
    tasks=task_names,
    num_fewshot=0,
    batch_size=64,
    task_manager=tm,
    limit=100,
    gen_kwargs={
        "until": ["<<END>>"],
        "max_gen_toks": 32,
        "temperature": 0.0,
        "do_sample": False,
    },
    use_cache=None,
    cache_requests=False,
    rewrite_requests_cache=False,
    delete_requests_cache=False,
    log_samples=False,
    write_out=False,
)






logger.info("Evaluation completed successfully")
logger.info(f"Metrics summary: {results['results']}")

# 4. Save results back to Drive
SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)

sparsity_tag = "0.00"
filename = f"results_s{sparsity_tag}_{'_'.join(task_names)}_squadv1.json"
result_path = os.path.join(SAVE_DIR, filename)

metrics_only = results["results"]

with open(result_path, "w") as f:
    json.dump(metrics_only, f, indent=2)

logger.info(f"Saved results to: {result_path}")
logger.info("Finished SQuADv1 evaluation script")


In [ ]:
import torch.nn.functional as F
def compute_answer_loglikelihood(model, tokenizer, prompt: str, answer: str, device: str) -> float:
    """
    Compute average log-likelihood of `answer` given `prompt` under the model.
    If answer == "", return 0.0 (the empty string baseline).
    """
    if answer == "":
        return 0.0

    # Encode prompt and answer separately
    enc_prompt = tokenizer(prompt, return_tensors="pt")
    enc_answer = tokenizer(answer, return_tensors="pt", add_special_tokens=False)

    input_ids = torch.cat(
        [enc_prompt["input_ids"], enc_answer["input_ids"]],
        dim=1
    ).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits  # (1, seq_len, vocab)

    # We predict token t+1 at position t
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # (1, seq_len-1, vocab)

    prefix_len = enc_prompt["input_ids"].shape[1]
    answer_ids = enc_answer["input_ids"][0].to(device)
    answer_len = answer_ids.shape[0]

    # Answer tokens are generated starting right after the prompt
    # First answer token is predicted at position prefix_len-1
    start = prefix_len - 1
    end = start + answer_len

    # Safety check: if anything goes weird with lengths, just bail gracefully
    if end > log_probs.shape[1]:
        end = log_probs.shape[1]
        answer_ids = answer_ids[: (end - start)]
        answer_len = answer_ids.shape[0]
        if answer_len == 0:
            return 0.0

    token_log_probs = log_probs[0, start:end, :]  # (answer_len, vocab)
    # Gather logprob of the correct answer token at each step
    gathered = token_log_probs.gather(-1, answer_ids.unsqueeze(-1)).squeeze(-1)

    # You can return sum or mean; mean reduces length bias a bit
    return gathered.mean().item()



In [ ]:
#squadv2 with "" unaswerable, not prompt suggestion
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-125m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 32
LIMIT = None  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/opt125_manual"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv2_manual_full.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD2.0 validation
# ----------------------------
dataset = load_dataset("squad_v2", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    # 1) Greedy generation of a candidate answer
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated, skip_special_tokens=True)

    # Optional: stop at first newline
    if "\n" in raw_text:
        candidate_answer = raw_text.split("\n", 1)[0].strip()
    else:
        candidate_answer = raw_text.strip()

    # 2) Compute log-likelihood of candidate vs empty answer ""
    ll_answer = compute_answer_loglikelihood(model, tokenizer, prompt, candidate_answer, DEVICE)
    ll_empty = compute_answer_loglikelihood(model, tokenizer, prompt, "", DEVICE)  # will be 0.0

    if ll_empty >= ll_answer:
        # Model "prefers" to give no answer
        pred_text = ""
    else:
        pred_text = candidate_answer

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("RAW GEN :", repr(raw_text))
        print("CAND   :", repr(candidate_answer))
        print("LL(answer) =", ll_answer, "  LL(empty) =", ll_empty)
        print("FINAL PRED:", repr(pred_text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": pred_text,
        "no_answer_probability": 0.0,  # we're not tuning threshold here
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })



# ----------------------------
# 5. Compute SQuAD2 metrics
# ----------------------------
metric = evaluate.load("squad_v2")  # SQuAD 2.0 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


In [ ]:
#working manual squadv2 eval function, this is the baseline to use!!!
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-125m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 32
LIMIT = None  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/opt125_manual"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv2_manual_full.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD2.0 validation
# ----------------------------
dataset = load_dataset("squad_v2", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nIf the question cannot be answered from the context, answer with 'unanswerable'."
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated, skip_special_tokens=True)

    if "\n" in raw_text:
        text = raw_text.split("\n", 1)[0].strip()
    else:
        text = raw_text.strip()

    # Simple heuristic: treat "unanswerable" as empty prediction
    text_lower = text.lower()
    if text_lower.startswith("unanswerable"):
        pred_text = ""
    else:
        pred_text = text

    # DEBUG
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("RAW PRED:", repr(text))
        print("PRED    :", repr(pred_text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": pred_text,
        "no_answer_probability": 0.0,  # more on this below
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })


# ----------------------------
# 5. Compute SQuAD2 metrics
# ----------------------------
metric = evaluate.load("squad_v2")  # SQuAD 2.0 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


INFO - NumExpr defaulting to 12 threads.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

==== DEBUG EXAMPLE ====
Q   : Who was the Norse leader?
GT  : ['Rollo', 'Rollo', 'Rollo', 'Rollo']
RAW PRED: 'The Norman king, Norman, was the first king of West Francia to give his name to Normandy. He was the first king of West Francia to give'
PRED    : 'The Norman king, Norman, was the first king of West Francia to give his name to Normandy. He was the first king of West Francia to give'


  0%|          | 5/11873 [00:02<1:06:36,  2.97it/s]

==== DEBUG EXAMPLE ====
Q   : What century did the Normans first gain their separate identity?
GT  : ['10th century', 'the first half of the 10th century', '10th', '10th']
RAW PRED: 'The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("'
PRED    : 'The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("'


  1%|          | 113/11873 [00:31<54:11,  3.62it/s]


KeyboardInterrupt: 

In [3]:
#working squadv1, use this as a baseline!!!
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-125m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 32
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual_full_\n.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    #text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    raw_text = tokenizer.decode(generated, skip_special_tokens=True)
    text = raw_text.split("\n", 1)[0].strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos won the Super Bowl 50, defeating the Carolina Panthers 24–10. The Broncos won the Super Bowl 50, defeating the Carolina Panthers 24–10'


  5%|▌         | 5/100 [00:02<00:34,  2.79it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th anniversary was chosen to emphasize the 50th anniversary of the Super Bowl. The color of the 50th anniversary was chosen to emphasize the'


100%|██████████| 100/100 [00:30<00:00,  3.30it/s]


Scores: {'exact_match': 13.0, 'f1': 25.07131408801484}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual_full_
.json


In [ ]:
#we put a max_tokens smaller in order to have more exact match
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-1.3b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 8
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda
Evaluating on 100 examples


  1%|          | 1/100 [00:00<00:15,  6.20it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the AFC at Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos.\n\nThe Denver'


  2%|▏         | 2/100 [00:00<00:15,  6.32it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the NFC at Super Bowl 50?
GT  : ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
PRED: 'The Carolina Panthers.\n\nThe Carolina'


...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos.\nThe Denver Broncos'


  5%|▌         | 5/100 [00:00<00:14,  6.48it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th Super Bowl'


100%|██████████| 100/100 [00:14<00:00,  6.82it/s]


Scores: {'exact_match': 2.0, 'f1': 36.4576479076479}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual.json


In [ ]:
#different prompt to get smaller answers
#we put a max_tokens smaller in order to have more exact match
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-1.3b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 8
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer with a short phrase (no explanation):"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda
Evaluating on 100 examples


  1%|          | 1/100 [00:00<00:16,  6.17it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the AFC at Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The answer is:'


  2%|▏         | 2/100 [00:00<00:15,  6.25it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the NFC at Super Bowl 50?
GT  : ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
PRED: 'The answer is:'


...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The answer is:'


  5%|▌         | 5/100 [00:00<00:14,  6.59it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th'


100%|██████████| 100/100 [00:14<00:00,  6.82it/s]


Scores: {'exact_match': 0.0, 'f1': 18.63181818181818}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual.json
